# Lab 6b: Spark on Databricks

This notebook runs on the serverless compute of Databricks Free Edition. Run the cells one at a time, from the top.

## Create the volume

A volume of Unity Catalog keeps the files. Change `CATALOG` if your workspace has no catalog named `workspace`: Catalog Explorer lists the catalogs you can use.

In [ ]:
CATALOG = "workspace"
SCHEMA = "big_data_labs"
VOLUME = "lab06b"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
DATA = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
print(DATA)

## Upload the files

Upload the files to the folders `flights`, `airports` and `gutenberg-en` of the volume, as the README explains, and check them:

In [ ]:
for folder in ["flights", "airports", "gutenberg-en"]:
    names = sorted(file.name for file in dbutils.fs.ls(f"{DATA}/{folder}"))
    print(folder, len(names), "files:", names[0], "...", names[-1])

## Spark Connect

The notebook talks to Spark through Spark Connect: `spark` is a client of a remote Spark, and it has no `SparkContext`. The RDD API, and the RDD WordCount of `wordcount.ipynb`, do not work here.

In [ ]:
print(type(spark).__module__)
try:
    spark.sparkContext
except Exception as error:  # pylint: disable=broad-except
    print(type(error).__name__)

## Read the flights

The schema is given explicitly.

In [ ]:
schema = ("flight_id long, flight_date date, airline_code string, "
          "flight_number string, tail_number string, origin string, "
          "destination string, scheduled_departure timestamp, "
          "actual_departure timestamp, distance_km int, cancelled int")

flights = spark.read.csv(f"{DATA}/flights", header=True, schema=schema)
flights.count()

## Transformations are lazy

These lines only note the plan: the cell finishes at once. The action of the next cell runs the whole plan.

In [ ]:
from pyspark.sql import functions as F

flown = flights.filter(F.col("cancelled") == 0).withColumn(
    "delay_min", (F.col("actual_departure").cast("long") - F.col("scheduled_departure").cast("long")) / 60)
late = flown.filter(F.col("delay_min") > 15)

In [ ]:
late.count()

## The shuffle

Grouping by origin needs a shuffle: the plan shows it as an exchange step.

In [ ]:
by_origin = flights.groupBy("origin").count()
by_origin.explain()
by_origin.orderBy(F.desc("count")).show(5)

## The same query in Spark SQL

In [ ]:
flights.createOrReplaceTempView("flights")
spark.sql("""
    SELECT origin, count(*) AS count
    FROM flights
    GROUP BY origin
    ORDER BY count DESC
""").show(5)

## A join with a small table

In [ ]:
airports = spark.read.csv(f"{DATA}/airports", header=True, inferSchema=True)
by_country = (flights.join(airports, flights.origin == airports.airport_code)
              .groupBy("country")
              .count()
              .orderBy(F.desc("count")))
by_country.show()

## A window function: running total of delay minutes per airline

In [ ]:
from pyspark.sql import Window

daily = flown.groupBy("airline_code", "flight_date").agg(F.sum("delay_min").alias("delay_min_total"))
running = Window.partitionBy("airline_code").orderBy("flight_date")
total = daily.withColumn("running_delay_min", F.sum("delay_min_total").over(running))
total.filter(F.col("airline_code") == "AND").orderBy("flight_date").show(5)

## WordCount with DataFrames

The same DataFrame version of `wordcount.ipynb`, which needs no RDD.

In [ ]:
books = spark.read.text(f"{DATA}/gutenberg-en")
df_counts = (books.select(F.explode(F.split(F.lower("value"), r"[^\p{L}]+")).alias("word"))
             .filter(F.col("word") != "")
             .groupBy("word")
             .count()
             .orderBy(F.desc("count"), "word"))
df_counts.show(10)

## Save a table

`saveAsTable` writes a managed table of Unity Catalog in the schema.

In [ ]:
by_country.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.flights_by_country")
spark.table(f"{CATALOG}.{SCHEMA}.flights_by_country").orderBy(F.desc("count")).show(3)

## Remove

Delete the table, the volume with its files, and the schema.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.flights_by_country")
spark.sql(f"DROP VOLUME IF EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA}")